In [0]:
# Import important libraries 
from pyspark.sql.functions import (
    col, row_number, monotonically_increasing_id, lit,
    year, month, dayofmonth, dayofweek, quarter, weekofyear,
    date_format, expr, sequence, explode, to_date, when, concat, lpad,
    max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, DateType
from delta.tables import DeltaTable

silver_table = "crimeworkspace.silver.silver_crime"
gold_schema = "crimeworkspace.gold"

In [0]:
# Handle incermental append.
try:
    # Check if we need to process any new dimension data
    silver_df = spark.read.table(silver_table)
    silver_watermark = silver_df.agg(spark_max("ingest_ts")).collect()[0][0]
    print(f"Latest silver data timestamp: {silver_watermark}")
except Exception as e:
    print(f"Cannot read silver table: {e}")
    dbutils.notebook.exit("No silver data")

Latest silver data timestamp: 2025-11-09 15:15:20.711000


# Dims Date and Time.


In [0]:
# Generate date range from 2020-01-01 to 2030-12-31
dim_date_table = f"{gold_schema}.dim_date"

try:
    # Check if Dim_Date already exists
    existing_dim_date = spark.read.table(dim_date_table)
    date_count = existing_dim_date.count()
    print(f"Dim_Date already exists with {date_count:,} rows - SKIPPING")
except:
    # Create Dim_Date only if it doesn't exist
    print("Creating Dim_Date for the first time...")
    
    date_df = spark.sql("""
        SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) as date
    """)
    
    dim_date = date_df.select(
        col("date").alias("date_key"),
        year(col("date")).alias("year"),
        quarter(col("date")).alias("quarter"),
        month(col("date")).alias("month"),
        dayofmonth(col("date")).alias("day"),
        weekofyear(col("date")).alias("week_of_year"),
        dayofweek(col("date")).alias("day_of_week"),
        date_format(col("date"), "EEEE").alias("day_name"),
        date_format(col("date"), "MMMM").alias("month_name"),
        when(dayofweek(col("date")).isin(1, 7), True).otherwise(False).alias("is_weekend"),
        when(month(col("date")) >= 7, year(col("date")) + 1).otherwise(year(col("date"))).alias("fiscal_year"),
        concat(lit("Q"), quarter(col("date"))).alias("quarter_name")
    )
    
    dim_date.write.format("delta").mode("overwrite").saveAsTable(dim_date_table)

# Dim_Time


dim_time_table = f"{gold_schema}.dim_time"

try:
    # Check if Dim_Time already exists
    existing_dim_time = spark.read.table(dim_time_table)
    time_count = existing_dim_time.count()
    print(f"Dim_Time already exists with {time_count:,} rows - SKIPPING")
except:
    # Create Dim_Time only if it doesn't exist
    print("Creating Dim_Time for the first time...")
    
    time_df = spark.range(0, 1440).select(col("id").alias("minute_of_day"))
    
    dim_time = time_df.select(
        concat(
            lpad((col("minute_of_day") / 60).cast("int").cast("string"), 2, "0"),
            lit(":"),
            lpad((col("minute_of_day") % 60).cast("int").cast("string"), 2, "0")
        ).alias("time_key"),
        (col("minute_of_day") / 60).cast("int").alias("hour"),
        (col("minute_of_day") % 60).cast("int").alias("minute"),
        when((col("minute_of_day") / 60).cast("int").between(6, 11), "Morning")
        .when((col("minute_of_day") / 60).cast("int").between(12, 17), "Afternoon")
        .when((col("minute_of_day") / 60).cast("int").between(18, 21), "Evening")
        .otherwise("Night").alias("time_period"),
        when((col("minute_of_day") / 60).cast("int") == 0, "12 AM")
        .when((col("minute_of_day") / 60).cast("int") < 12, 
              concat((col("minute_of_day") / 60).cast("int").cast("string"), lit(" AM")))
        .when((col("minute_of_day") / 60).cast("int") == 12, "12 PM")
        .otherwise(
              concat(((col("minute_of_day") / 60).cast("int") - 12).cast("string"), lit(" PM"))
        ).alias("hour_12_format")
    )
    
    dim_time.write.format("delta").mode("overwrite").saveAsTable(dim_time_table)

Creating Dim_Date for the first time...
Creating Dim_Time for the first time...


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_date

date_key,year,quarter,month,day,week_of_year,day_of_week,day_name,month_name,is_weekend,fiscal_year,quarter_name
2020-01-01,2020,1,1,1,1,4,Wednesday,January,false,2020,Q1
2020-01-02,2020,1,1,2,1,5,Thursday,January,false,2020,Q1
2020-01-03,2020,1,1,3,1,6,Friday,January,false,2020,Q1
2020-01-04,2020,1,1,4,1,7,Saturday,January,true,2020,Q1
2020-01-05,2020,1,1,5,1,1,Sunday,January,true,2020,Q1
2020-01-06,2020,1,1,6,2,2,Monday,January,false,2020,Q1
2020-01-07,2020,1,1,7,2,3,Tuesday,January,false,2020,Q1
2020-01-08,2020,1,1,8,2,4,Wednesday,January,false,2020,Q1
2020-01-09,2020,1,1,9,2,5,Thursday,January,false,2020,Q1
2020-01-10,2020,1,1,10,2,6,Friday,January,false,2020,Q1


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_time

time_key,hour,minute,time_period,hour_12_format
09:00,9,0,Morning,9 AM
09:01,9,1,Morning,9 AM
09:02,9,2,Morning,9 AM
09:03,9,3,Morning,9 AM
09:04,9,4,Morning,9 AM
09:05,9,5,Morning,9 AM
09:06,9,6,Morning,9 AM
09:07,9,7,Morning,9 AM
09:08,9,8,Morning,9 AM
09:09,9,9,Morning,9 AM


# Dim Police Station.

In [0]:
df_silver = spark.read.table(silver_table)

dim_police_station_table = f"{gold_schema}.dim_police_station"

# Get new distinct police stations from silver
new_stations = df_silver.select(
    "area_code",
    "area_name",
    "reporting_district"
).distinct()

try:
    # Table exists - perform MERGE
    existing_stations = DeltaTable.forName(spark, dim_police_station_table)
    
    # Find stations that don't exist yet
    new_stations_to_add = new_stations.join(
        spark.read.table(dim_police_station_table).select("area_code", "reporting_district"),
        ["area_code", "reporting_district"],
        "leftanti"
    )
    
    new_count = new_stations_to_add.count()
    
    if new_count > 0:
        print(f"Found {new_count} new stations to add")
        
        # Get max existing key
        max_key = spark.read.table(dim_police_station_table).agg(spark_max("station_key")).collect()[0][0]
        if max_key is None:
            max_key = 0
        
        # Add surrogate keys to new records
        window_spec = Window.orderBy("area_code", "reporting_district")
        new_stations_with_keys = new_stations_to_add.withColumn(
            "station_key",
            row_number().over(window_spec) + max_key
        ).select("station_key", "area_code", "area_name", "reporting_district")
        
        # Append new stations
        new_stations_with_keys.write.format("delta").mode("append").saveAsTable(dim_police_station_table)
        print(f"Added {new_count} new stations")
    else:
        print("No new stations - SKIPPING")
    
except:
    # Table doesn't exist - CREATE
    print("Creating Dim_Police_Station for the first time...")
    
    window_spec = Window.orderBy("area_code", "reporting_district")
    dim_police_station = new_stations.withColumn(
        "station_key",
        row_number().over(window_spec)
    ).select("station_key", "area_code", "area_name", "reporting_district")
    
    dim_police_station.write.format("delta").mode("overwrite").saveAsTable(dim_police_station_table)
    station_count = dim_police_station.count()
    print(f"Dim_Police_Station created: {station_count:,} stations")


Creating Dim_Police_Station for the first time...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_Police_Station created: 1,154 stations


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_police_station

station_key,area_code,area_name,reporting_district
1,1,Central,101
2,1,Central,105
3,1,Central,109
4,1,Central,111
5,1,Central,112
6,1,Central,118
7,1,Central,119
8,1,Central,121
9,1,Central,122
10,1,Central,123


# Dim Crime Location

In [0]:
dim_location_table = f"{gold_schema}.dim_location"

# Get new distinct locations from silver
new_locations = df_silver.select(
    "premise_code",
    "premise_description",
    "premise_category",
    "latitude",
    "longitude"
).distinct()

try:
    # Table exists - perform MERGE
    existing_locations = spark.read.table(dim_location_table)
    
    # Find locations that don't exist yet
    new_locations_to_add = new_locations.join(
        existing_locations.select("premise_code", "latitude", "longitude"),
        ["premise_code", "latitude", "longitude"],
        "leftanti"
    )
    
    new_count = new_locations_to_add.count()
    
    if new_count > 0:
        print(f"Found {new_count} new locations to add")
        
        # Get max existing key
        max_key = existing_locations.agg(spark_max("location_key")).collect()[0][0]
        if max_key is None:
            max_key = 0
        
        # Add surrogate keys to new records
        window_spec = Window.orderBy("premise_code", "latitude", "longitude")
        new_locations_with_keys = new_locations_to_add.withColumn(
            "location_key",
            row_number().over(window_spec) + max_key
        ).select("location_key", "premise_code", "premise_description", "premise_category", "latitude", "longitude")
        
        # Append new locations
        new_locations_with_keys.write.format("delta").mode("append").saveAsTable(dim_location_table)
        print(f"Added {new_count} new locations")
    else:
        print("No new locations - SKIPPING")
    
except:
    # Table doesn't exist - CREATE
    print("Creating Dim_Location for the first time...")
    
    window_spec = Window.orderBy("premise_code", "latitude", "longitude")
    dim_location = new_locations.withColumn(
        "location_key",
        row_number().over(window_spec)
    ).select("location_key", "premise_code", "premise_description", "premise_category", "latitude", "longitude")
    
    dim_location.write.format("delta").mode("overwrite").saveAsTable(dim_location_table)
    location_count = dim_location.count()
    print(f"Dim_Location created: {location_count:,} locations")

Creating Dim_Location for the first time...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_Location created: 109,468 locations


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_location

location_key,premise_code,premise_description,premise_category,latitude,longitude
1,-999,Unspecified,Unspecified,33.7801,-118.2611
2,-999,Unspecified,Unspecified,33.7805,-118.2533
3,-999,Unspecified,Unspecified,33.7879,-118.3075
4,-999,Unspecified,Unspecified,33.931,-118.2916
5,-999,Unspecified,Unspecified,33.9519,-118.2781
6,-999,Unspecified,Unspecified,33.9529,-118.3961
7,-999,Unspecified,Unspecified,33.96,-118.3141
8,-999,Unspecified,Unspecified,33.9656,-118.2651
9,-999,Unspecified,Unspecified,33.9682,-118.309
10,-999,Unspecified,Unspecified,33.9713,-118.2739


# Dim Crime Type

In [0]:
dim_crime_type_table = f"{gold_schema}.dim_crime_type"

# Get new distinct crime types from silver
new_crime_types = df_silver.select(
    "crime_code",
    "crime_description",
    "crime_category",
    "crime_part"
).distinct()

try:
    # Table exists - perform MERGE
    existing_crime_types = spark.read.table(dim_crime_type_table)
    
    # Find crime types that don't exist yet
    new_types_to_add = new_crime_types.join(
        existing_crime_types.select("crime_code"),
        ["crime_code"],
        "leftanti"
    )
    
    new_count = new_types_to_add.count()
    
    if new_count > 0:
        print(f"Found {new_count} new crime types to add")
        
        # Get max existing key
        max_key = existing_crime_types.agg(spark_max("crime_type_key")).collect()[0][0]
        if max_key is None:
            max_key = 0
        
        # Add surrogate keys to new records
        window_spec = Window.orderBy("crime_code")
        new_types_with_keys = new_types_to_add.withColumn(
            "crime_type_key",
            row_number().over(window_spec) + max_key
        ).select("crime_type_key", "crime_code", "crime_description", "crime_category", "crime_part")
        
        # Append new crime types
        new_types_with_keys.write.format("delta").mode("append").saveAsTable(dim_crime_type_table)
        print(f"Added {new_count} new crime types")
    else:
        print("No new crime types - SKIPPING")
    
except:
    # Table doesn't exist - CREATE
    print("Creating Dim_Crime_Type for the first time...")
    
    window_spec = Window.orderBy("crime_code")
    dim_crime_type = new_crime_types.withColumn(
        "crime_type_key",
        row_number().over(window_spec)
    ).select("crime_type_key", "crime_code", "crime_description", "crime_category", "crime_part")
    
    dim_crime_type.write.format("delta").mode("overwrite").saveAsTable(dim_crime_type_table)
    crime_type_count = dim_crime_type.count()
    print(f"Dim_Crime_Type created: {crime_type_count:,} crime types")

Creating Dim_Crime_Type for the first time...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_Crime_Type created: 129 crime types


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_crime_type

crime_type_key,crime_code,crime_description,crime_category,crime_part
1,110,CRIMINAL HOMICIDE,Homicide,1
2,113,"MANSLAUGHTER, NEGLIGENT",Homicide,1
3,121,"RAPE, FORCIBLE",Sex Crimes & Child Abuse,1
4,122,"RAPE, ATTEMPTED",Sex Crimes & Child Abuse,1
5,210,ROBBERY,Theft & Burglary,1
6,220,ATTEMPTED ROBBERY,Theft & Burglary,1
7,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",Assault & Battery,1
8,231,ASSAULT WITH DEADLY WEAPON ON POLICE OFFICER,Assault & Battery,1
9,235,CHILD ABUSE (PHYSICAL) - AGGRAVATED ASSAULT,Assault & Battery,1
10,236,INTIMATE PARTNER - AGGRAVATED ASSAULT,Assault & Battery,1


# Dim Victim

In [0]:
dim_victim_table = f"{gold_schema}.dim_victim"

# Get new distinct victims from silver
new_victims = df_silver.select(
    "victim_age",
    "victim_sex",
    "victim_descent"
).distinct()

# Add age group
new_victims = new_victims.withColumn(
    "age_group",
    when(col("victim_age").isNull(), "Unknown")
    .when(col("victim_age") < 13, "Child (0-12)")
    .when(col("victim_age").between(13, 17), "Teen (13-17)")
    .when(col("victim_age").between(18, 24), "Young Adult (18-24)")
    .when(col("victim_age").between(25, 44), "Adult (25-44)")
    .when(col("victim_age").between(45, 54), "Middle Age (45-54)")
    .when(col("victim_age").between(55, 64), "Senior (55-64)")
    .when(col("victim_age") >= 65, "Elderly (65+)")
    .otherwise("Unknown")
)

try:
    # Table exists - perform MERGE
    existing_victims = spark.read.table(dim_victim_table)
    
    # Find victims that don't exist yet
    new_victims_to_add = new_victims.join(
        existing_victims.select("victim_age", "victim_sex", "victim_descent"),
        ["victim_age", "victim_sex", "victim_descent"],
        "leftanti"
    )
    
    new_count = new_victims_to_add.count()
    
    if new_count > 0:
        print(f"Found {new_count} new victim profiles to add")
        
        # Get max existing key
        max_key = existing_victims.agg(spark_max("victim_key")).collect()[0][0]
        if max_key is None:
            max_key = 0
        
        # Add surrogate keys to new records
        window_spec = Window.orderBy("victim_age", "victim_sex", "victim_descent")
        new_victims_with_keys = new_victims_to_add.withColumn(
            "victim_key",
            row_number().over(window_spec) + max_key
        ).select("victim_key", "victim_age", "age_group", "victim_sex", "victim_descent")
        
        # Append new victims
        new_victims_with_keys.write.format("delta").mode("append").saveAsTable(dim_victim_table)
        print(f"Added {new_count} new victim profiles")
    else:
        print("No new victim profiles - SKIPPING")
    
except:
    # Table doesn't exist - CREATE
    print("Creating Dim_Victim for the first time...")
    
    window_spec = Window.orderBy("victim_age", "victim_sex", "victim_descent")
    dim_victim = new_victims.withColumn(
        "victim_key",
        row_number().over(window_spec)
    ).select("victim_key", "victim_age", "age_group", "victim_sex", "victim_descent")
    
    dim_victim.write.format("delta").mode("overwrite").saveAsTable(dim_victim_table)
    victim_count = dim_victim.count()
    print(f"Dim_Victim created: {victim_count:,} victim profiles")

Creating Dim_Victim for the first time...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_Victim created: 1,810 victim profiles


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_victim

victim_key,victim_age,age_group,victim_sex,victim_descent
1,0,Child (0-12),Female,Black
2,0,Child (0-12),Female,Korean
3,0,Child (0-12),Female,Latin
4,0,Child (0-12),Female,Other
5,0,Child (0-12),Female,Other Asian
6,0,Child (0-12),Female,Unknown
7,0,Child (0-12),Female,White
8,0,Child (0-12),Male,Black
9,0,Child (0-12),Male,Hawaiian
10,0,Child (0-12),Male,Korean


# Dim Weapon

In [0]:
dim_weapon_table = f"{gold_schema}.dim_weapon"

# Get new distinct weapons from silver
new_weapons = df_silver.select(
    "weapon_code",
    "weapon_description"
).distinct()

try:
    # Table exists - perform MERGE
    existing_weapons = spark.read.table(dim_weapon_table)
    
    # Find weapons that don't exist yet
    new_weapons_to_add = new_weapons.join(
        existing_weapons.select("weapon_code"),
        ["weapon_code"],
        "leftanti"
    )
    
    new_count = new_weapons_to_add.count()
    
    if new_count > 0:
        print(f"Found {new_count} new weapon types to add")
        
        # Get max existing key
        max_key = existing_weapons.agg(spark_max("weapon_key")).collect()[0][0]
        if max_key is None:
            max_key = 0
        
        # Add surrogate keys to new records
        window_spec = Window.orderBy("weapon_code")
        new_weapons_with_keys = new_weapons_to_add.withColumn(
            "weapon_key",
            row_number().over(window_spec) + max_key
        ).select("weapon_key", "weapon_code", "weapon_description")
        
        # Append new weapons
        new_weapons_with_keys.write.format("delta").mode("append").saveAsTable(dim_weapon_table)
        print(f"Added {new_count} new weapon types")
    else:
        print("No new weapons - SKIPPING")
    
except:
    # Table doesn't exist - CREATE
    print("Creating Dim_Weapon for the first time...")
    
    window_spec = Window.orderBy("weapon_code")
    dim_weapon = new_weapons.withColumn(
        "weapon_key",
        row_number().over(window_spec)
    ).select("weapon_key", "weapon_code", "weapon_description")
    
    dim_weapon.write.format("delta").mode("overwrite").saveAsTable(dim_weapon_table)
    weapon_count = dim_weapon.count()
    print(f"Dim_Weapon created: {weapon_count:,} weapon types")

Creating Dim_Weapon for the first time...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_Weapon created: 78 weapon types


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.dim_weapon

weapon_key,weapon_code,weapon_description
1,0,No Weapon
2,101,REVOLVER
3,102,HAND GUN
4,103,RIFLE
5,104,SHOTGUN
6,105,SAWED OFF RIFLE/SHOTGUN
7,106,UNKNOWN FIREARM
8,107,OTHER FIREARM
9,108,AUTOMATIC WEAPON/SUB-MACHINE GUN
10,109,SEMI-AUTOMATIC PISTOL
